# Compare the models' segmentation on Google Colab

Open this notebook from its GitHub link, so the cells are always those on `main`:
<https://colab.research.google.com/github/meiazero/garimpo-uncertainty/blob/main/notebooks/02_compare_segmentation.ipynb>

It draws `qualitative_cells` from the finished runs in Drive, on the real Sentinel-2 cells of the
unpacked store in `MyDrive/garimpo/garimpo-dataset/`. Each row is one test cell, chosen from the reference alone before
any model is read: the first empty cell, then the mining cells at the quartiles and the maximum of
reference area. The columns are its true color, its reference mask, and each model's mining
probability with the reference outlined in white, the predicted area and the intervals of the
conformal simple (CS) and COMPASS-J PCA (CJ) under it.

1. Set the form in section 3: the scale and the models to compare.
2. Run Runtime > Run all.
3. Look at the figure at the end of section 4; its PDF and SVG are in `MyDrive/garimpo/figures/`.

A CPU runtime is enough: the figure reads the runs' stored predictions, never a checkpoint. The
notebook only reads `MyDrive/garimpo/runs/`; it writes to `MyDrive/garimpo/figures/` alone.

| Section | What it does |
|---|---|
| 1. Setup | Mounts Drive, clones the code, builds the environment with the figure dependencies |
| 2. Data | Links the store and the runs in Drive, copying neither |
| 3. What to draw | The form, and the runs the figure will read |
| 4. Figure | Draws the figure, shows it, copies it to Drive |
| 5. Release | Releases the runtime |

## 1. Setup

The runtime's disk is erased when a session ends, so this cell clones the code and installs
`requirements.txt` again, unless pip would replace Colab's torch or a CUDA library: the install
then stops and names it. `DATA` is the unpacked store in Drive, which the figure reads in place.

In [ ]:
# @title Setup
import json
import os
import subprocess
from pathlib import Path

import IPython.display
from google.colab import drive, runtime

REPO = "https://github.com/meiazero/garimpo-uncertainty.git"
CODE = "/content/garimpo-uncertainty"  # the clone; erased with the session
DRIVE = "/content/drive/MyDrive/garimpo"  # kept between sessions
DATA = f"{DRIVE}/garimpo-dataset"  # the unpacked store, read in place
os.environ["GARIMPO_DATA_DIR"] = DATA


def release_runtime() -> None:
    """Write pending files to Drive, then release the runtime, which ends the session."""
    try:
        drive.flush_and_unmount()
    except Exception as error:  # noqa: BLE001 -- the runtime must be released whatever failed
        print(f"Drive did not confirm the last writes; check the files in Drive: {error}")
    runtime.unassign()


drive.mount("/content/drive")
!apt-get -qq install -y rsync poppler-utils > /dev/null
if not Path(CODE, ".git").is_dir():
    !git clone --recurse-submodules {REPO} {CODE}
%cd {CODE}
!python scripts/colab_install.py
!git log -1 --format='Drawing with commit %h %cs %s'

## 2. Data

Nothing is copied: `runs` in the clone is a link to `MyDrive/garimpo/runs/`, and
`GARIMPO_DATA_DIR` is `MyDrive/garimpo/garimpo-dataset/`. The figure reads the sidecars in `tiles/`
and, of `images.npy` and `masks.npy`, only the cells it shows; `scl.npy`, `zenodo/` and `weights/`
are never opened. It writes only to `outputs/` in the clone.

In [ ]:
# @title Link the store and the runs in Drive
for needed in ("tiles", "images.npy", "masks.npy"):
    if not Path(DATA, needed).exists():
        print(f"Missing {DATA}/{needed}: the figure cannot draw without it.")
if not os.path.lexists("runs"):  # a link to a missing folder still exists
    !ln -s {DRIVE}/runs runs
!ls runs

## 3. What to draw

- `SCALE`: `pilot` for the runs on 1% of the training tiles, `full` for the experiment. `auto`
  takes the full runs once one has a checkpoint, and the pilot otherwise.
- The model boxes: the columns of the figure, in this order. A model whose run has no intervals yet
  is left out, with a message.

In [ ]:
# @title The form
SCALE = "auto"  # @param ["auto", "pilot", "full"]
UNET_SCRATCH = True  # @param {type:"boolean"}
UNET_IMAGENET = True  # @param {type:"boolean"}
UNET_SENTINEL2 = True  # @param {type:"boolean"}
CONVNET = True  # @param {type:"boolean"}

MODELS = [
    model
    for model, chosen in (
        ("unet_scratch", UNET_SCRATCH),
        ("unet_imagenet", UNET_IMAGENET),
        ("unet_sentinel2", UNET_SENTINEL2),
        ("convnet", CONVNET),
    )
    if chosen
]
try:
    chosen_scale = subprocess.run(  # noqa: S603
        ["garimpo", "status", SCALE, "--choose"],  # noqa: S607
        capture_output=True,
        text=True,
        check=False,
    )
    if chosen_scale.returncode == 0:
        SCALE = chosen_scale.stdout.splitlines()[-1]
    else:
        print(chosen_scale.stdout, chosen_scale.stderr)
except OSError as error:
    # A raise here would cancel the release queued at the end; left at auto, no run is found.
    print(f"Could not choose the scale: {error!r}")
CONFIG = Path("configs/scales", f"{SCALE}.json")
scale = json.loads(CONFIG.read_text(encoding="utf-8")) if CONFIG.exists() else {}
RUNS = []
for model in MODELS if scale else []:
    # The run directory as RunConfig.run_dir writes it: runs/pilot/unet_scratch-f0.01.
    run = Path(scale["RUNS_DIR"], f"{model}-f{scale['TRAIN_FRACTION']}")
    if Path(run, "intervals", "intervals.npz").exists():
        RUNS.append(str(run))
    else:
        print(f"Left out {model}: {run} has no intervals yet.")
OUT = f"outputs/figures-{SCALE}"
print(f"Scale {SCALE}; runs {', '.join(RUNS) or 'none'}.")

## 4. Figure

Draws the figure into `outputs/figures-<scale>/`, then copies that folder to
`MyDrive/garimpo/figures/<scale>/`. The audit line printed after the figure must read `clean`.
On the pilot the models trained for 40 updates, so the figure checks the drawing, not the
models.

In [ ]:
# @title Draw the segmentation comparison
FIGURE = Path(OUT, "qualitative_cells")
RUN_DIRS = " ".join(RUNS)
for suffix in (".pdf", ".svg", ".png"):
    FIGURE.with_suffix(suffix).unlink(missing_ok=True)  # an earlier Run all's figure is never shown
if RUNS:
    !garimpo figures {RUN_DIRS} --out {OUT} --only qualitative_cells
    !pdftoppm -png -r 150 -singlefile {FIGURE}.pdf {FIGURE}
    !mkdir -p {DRIVE}/figures/{SCALE}
    !rsync -a {OUT}/ {DRIVE}/figures/{SCALE}/
else:
    print("Skipped: no chosen model has a run with intervals.")
if FIGURE.with_suffix(".png").exists():
    IPython.display.display(IPython.display.Image(filename=str(FIGURE.with_suffix(".png"))))

## 5. Release

Writes the pending files to Drive and releases the runtime. No cell before this one raises a Python
error, so Run all always reaches it; the figure above stays in the notebook's output.

In [ ]:
# @title Release the runtime
release_runtime()